# Caso 01 — BrightCart (Moda Online): Limpieza de Datos y Primeros KPIs
## Día 2 — La *mise en place* del analista y la pregunta de Diego: «¿Bruto o neto?»

En este segundo cuaderno transformamos las tablas crudas en activos limpios y consistentes (`data/clean/`) y calculamos los primeros KPIs globales y comparativos de la campaña.

### Objetivos del Día 2:
1. **Mañana (Limpieza):** Eliminar duplicados de exportación, convertir `precio_unitario` a `float64`, normalizar `descuento_pct` a escala `[0.0, 1.0]`, excluir cantidades `<= 0`, unificar fechas mixtas a `datetime`, filtrar las `42` devoluciones huérfanas y estandarizar tallas y ciudades.
2. **Tarde (Primeros KPIs):** Resolver la discrepancia entre **Ingreso a precio de catálogo** vs **Ingreso real pagado en caja**, y medir el **efecto base** comparando la **Semana Previa sin promoción (`24/06`–`30/06`)** contra el arranque de las **Rebajas de Verano (`01/07`–`07/07`)** y el total de la campaña.

In [1]:
# 1. Importación de librerías y carga de datos base
import os
import pandas as pd
import numpy as np

lineas = pd.read_csv('data/lineas_pedido.csv')
pedidos = pd.read_csv('data/pedidos.csv')
productos = pd.read_csv('data/productos.csv')
devoluciones = pd.read_csv('data/devoluciones.csv')
campana = pd.read_csv('data/campana.csv')
clientes = pd.read_csv('data/clientes.csv')
visitas = pd.read_csv('data/visitas_web.csv')

print('Ficheros cargados correctamente.')
print('Filas iniciales en lineas:', len(lineas))

Ficheros cargados correctamente.
Filas iniciales en lineas: 20140


### Análisis de la Carga Inicial
* Partimos de los **`20.140` registros** en `lineas_pedido.csv` tal como salieron del exportador transaccional. A continuación aplicamos paso a paso las reglas de saneamiento.

---
## Reto 1 (Día 2): Depurar Duplicados en la Tabla de Hechos
En el Día 1 detectamos que `lineas_pedido` tenía `20.140` filas por un fallo del exportador SQL. Eliminamos las filas idénticas con `.drop_duplicates()`.

In [2]:
# =============================================================================
# RETO 1 (DÍA 2): ELIMINAR DUPLICADOS EN LINEAS_PEDIDO
# =============================================================================
lineas = lineas.drop_duplicates()
print('Total filas tras eliminar duplicados exactos:', len(lineas))
print('Duplicados exactos restantes:', lineas.duplicated().sum())

Total filas tras eliminar duplicados exactos: 20001
Duplicados exactos restantes: 0


### Análisis del Reto 1 (Deduplicación)
* Se han eliminado **`139` filas idénticas**, reduciendo la tabla de `20.140` a **`20.001` filas** sin ningún duplicado exacto restante.

---
## Reto 2 (Día 2): Limpieza de Precios Unitarios (`str` $ightarrow$ `float64`)
Traducimos `precio_unitario` del formato humano localizado (`"63,83 €"`) al formato numérico de cálculo (`63.83`).

In [3]:
# =============================================================================
# RETO 2 (DÍA 2): LIMPIEZA DE PRECIOS UNITARIOS
# =============================================================================
lineas['precio_unitario'] = (
    lineas['precio_unitario']
    .str.replace('€', '')
    .str.replace(',', '.')
    .str.strip()
    .astype(float)
)

print('Nuevo tipo de dato:', lineas['precio_unitario'].dtype)
print('\nResumen estadístico de precios:')
display(lineas['precio_unitario'].describe())

Nuevo tipo de dato: float64

Resumen estadístico de precios:


count    20001.000000
mean        37.308854
std         17.245409
min         10.360000
25%         25.060000
50%         34.450000
75%         48.410000
max         89.350000
Name: precio_unitario, dtype: float64

### Análisis del Reto 2 (`precio_unitario`)
* La columna ya es `float64` sin valores nulos. El precio unitario de catálogo oscila entre un **mínimo de `10,36 €`** y un **máximo de `89,35 €`**, con una **media de `37,31 €`** y una **mediana de `34,45 €`**.

---
## Reto 3 (Día 2): Normalizar Descuentos a Escala `[0.0 - 1.0]`
Unificamos las tres representaciones del descuento (`"20%"`, `0.2`, `20`) en una única fracción decimal entre `0.0` y `1.0`.

In [4]:
# =============================================================================
# RETO 3 (DÍA 2): NORMALIZAR DESCUENTOS A ESCALA 0.0 - 1.0
# =============================================================================
def normalizar_descuento(x):
    s = str(x).strip()
    if s.endswith('%'):
        return float(s[:-1]) / 100.0
    v = float(s)
    return v / 100.0 if v > 1.0 else v

lineas['descuento_pct'] = lineas['descuento_pct'].apply(normalizar_descuento)
print('Tipo de dato:', lineas['descuento_pct'].dtype)
print('\nValores únicos de descuento normalizados:')
print([float(x) for x in sorted(lineas['descuento_pct'].unique())])

Tipo de dato: float64

Valores únicos de descuento normalizados:
[0.0, 0.1, 0.2, 0.3, 0.4, 0.5]


### Análisis del Reto 3 (`descuento_pct`)
* La comprobación de valores únicos confirma que solo existen **6 escalones limpios de promoción**: `0%`, `10%`, `20%`, `30%`, `40%` y `50%` (`[0.0, 0.1, 0.2, 0.3, 0.4, 0.5]`), eliminando el riesgo de multiplicar precios por 20 o 40.

---
## Reto 4 (Día 2): Depurar Cantidades Imposibles (`cantidad <= 0`)
En una tabla de ventas, las cantidades nulas o negativas (`0`, `-1`, `-2`) son errores de registro que restarían unidades e ingresos si no se excluyen.

In [5]:
# =============================================================================
# RETO 4 (DÍA 2): DEPURAR CANTIDADES IMPOSIBLES
# =============================================================================
print('Valores únicos en cantidad antes de filtrar:', [int(x) for x in sorted(lineas['cantidad'].unique())])
filas_invalidas = (lineas['cantidad'] <= 0).sum()
print('Total filas imposibles (<= 0):', int(filas_invalidas))
print('\nMuestra de filas con cantidad <= 0:')
display(lineas[lineas['cantidad'] <= 0].head())

lineas = lineas[lineas['cantidad'] > 0].copy()
print('\nTotal filas legítimas en lineas_pedido:', len(lineas))
print('Valores únicos en cantidad tras filtrar:', [int(x) for x in sorted(lineas['cantidad'].unique())])

Valores únicos en cantidad antes de filtrar: [-1, 0, 1, 2, 3]
Total filas imposibles (<= 0): 15

Muestra de filas con cantidad <= 0:


,linea_id,pedido_id,producto_id,talla,cantidad,precio_unitario,descuento_pct
51,900052,500023,1234,L,-1,26.50,0.2
2139,902140,500833,1358,L,-1,26.83,0.2
6720,906721,502682,1203,M,0,19.01,0.1
6830,906831,502723,1176,m,0,23.41,0.2
7143,907144,502839,1123,41,0,77.67,0.3



Total filas legítimas en lineas_pedido: 19986
Valores únicos en cantidad tras filtrar: [1, 2, 3]


### Análisis del Reto 4 (Cantidades Válidas)
* Se detectaron y excluyeron **`15` líneas corruptas** con cantidades `<= 0` (`-2`, `-1`, `0`), quedando exactamente **`19.986` líneas de venta legítimas** con cantidades de `1`, `2` o `3` unidades por línea.

---
## Reto 5 (Día 2): Conversión de Fechas Mixtas a `datetime`
Convertimos `fecha_pedido` y `fecha_devolucion` al tipo `datetime64` usando `format='mixed'` y `dayfirst=True` para respetar el formato europeo `DD/MM/YYYY`.

In [6]:
# =============================================================================
# RETO 5 (DÍA 2): CONVERSIÓN DE FECHAS A DATETIME
# =============================================================================
pedidos['fecha_pedido'] = pd.to_datetime(pedidos['fecha_pedido'], format='mixed', dayfirst=True)
devoluciones['fecha_devolucion'] = pd.to_datetime(devoluciones['fecha_devolucion'], format='mixed', dayfirst=True)

print('Tipo de dato fecha_pedido:', pedidos['fecha_pedido'].dtype)
print('Tipo de dato fecha_devolucion:', devoluciones['fecha_devolucion'].dtype)
print('\nFecha mínima en pedidos:', pedidos['fecha_pedido'].min())
print('Fecha máxima en pedidos:', pedidos['fecha_pedido'].max())
print('\nNulos en fechas de pedidos:', pedidos['fecha_pedido'].isnull().sum())
print('Nulos en fechas de devoluciones:', devoluciones['fecha_devolucion'].isnull().sum())

Tipo de dato fecha_pedido: datetime64[us]
Tipo de dato fecha_devolucion: datetime64[us]

Fecha mínima en pedidos: 2024-01-07 00:00:00
Fecha máxima en pedidos: 2024-12-07 00:00:00

Nulos en fechas de pedidos: 0
Nulos en fechas de devoluciones: 0


### Análisis del Reto 5 (Fechas Estandarizadas)
* El `100 %` de las fechas en `pedidos` y `devoluciones` se ha convertido a `datetime64` con **`0` valores nulos (`NaT`)**, permitiendo filtrar con precisión la ventana de la Semana Previa (`2024-06-24` a `2024-06-30`) y la de Rebajas de Verano (`2024-07-01` a `2024-07-21`).

---
## Reto 6 (Día 2): Filtrar Devoluciones Huérfanas
Aislamos y descartamos las `42` devoluciones que apuntaban a `linea_id` inexistentes para garantizar la integridad referencial al cruzar ventas y retornos.

In [7]:
# =============================================================================
# RETO 6 (DÍA 2): FILTRAR DEVOLUCIONES HUÉRFANAS
# =============================================================================
devoluciones = devoluciones[devoluciones['linea_id'].isin(lineas['linea_id'])].copy()
print('Total devoluciones válidas y vinculadas a ventas:', len(devoluciones))

Total devoluciones válidas y vinculadas a ventas: 5516


### Análisis del Reto 6 (`devoluciones` limpias)
* Al filtrar contra las `19.986` líneas válidas, conservamos las **`5.516` devoluciones legítimas** y eliminamos las `42` huérfanas que habrían distorsionado la tasa de retorno.

---
## Reto 7 (Día 2): Estandarizar Tallas y Ciudades
Normalizamos las cadenas de texto en `talla` (quitando el prefijo `'Talla '` y pasando a mayúsculas) y en `ciudad` (formato título).

In [8]:
# =============================================================================
# RETO 7 (DÍA 2): ESTANDARIZAR TALLAS Y CIUDADES
# =============================================================================
lineas['talla'] = lineas['talla'].str.replace('Talla ', '', case=False).str.strip().str.upper()
devoluciones['talla'] = devoluciones['talla'].str.replace('Talla ', '', case=False).str.strip().str.upper()
pedidos['ciudad'] = pedidos['ciudad'].str.strip().str.title()

print('Tallas únicas normalizadas en ventas:')
print(sorted(lineas['talla'].unique()))
print('\nCiudades únicas normalizadas en pedidos:')
print(sorted(pedidos['ciudad'].unique()))

Tallas únicas normalizadas en ventas:
['36', '37', '38', '39', '40', '41', '42', 'L', 'M', 'S', 'XL', 'XS']

Ciudades únicas normalizadas en pedidos:
['A Coruña', 'Barcelona', 'Bilbao', 'Lisboa', 'Madrid', 'Malaga', 'Málaga', 'Oporto', 'Sevilla', 'Valencia', 'Zaragoza']


### Análisis del Reto 7 (Estandarización de Tallas y Ciudades)
* Las tallas quedan reducidas a **12 valores canónicos**: `36` a `42` para calzado y `XS`, `S`, `M`, `L`, `XL` para textil. Esto será decisivo en el Día 4 para descubrir en qué tallas exactas se concentra el problema de patronaje.

---
## Reto 8 (Día 2 — Tarde): Cálculo de los 4 KPIs Globales del Dataset
Con los datos limpios, calculamos la columna `importe_linea = cantidad * precio_unitario * (1 - descuento_pct)` y obtenemos los 4 KPIs base de todo el fichero exportado (Semana Previa + Campaña de Verano).

In [9]:
# =============================================================================
# RETO 8 (DÍA 2): CÁLCULO DE LOS 4 KPIs GLOBALES
# =============================================================================
lineas['importe_linea'] = lineas['cantidad'] * lineas['precio_unitario'] * (1 - lineas['descuento_pct'])

ingresos_totales = lineas['importe_linea'].sum()
unidades_totales = lineas['cantidad'].sum()
total_pedidos = lineas['pedido_id'].nunique()
ticket_medio = ingresos_totales / total_pedidos

print(f'Ingresos Totales (Pagados): {ingresos_totales:,.2f} €')
print(f'Unidades Vendidas:          {unidades_totales:,}')
print(f'Total Pedidos Válidos:      {total_pedidos:,}')
print(f'Ticket Medio Global:        {ticket_medio:,.2f} €')

Ingresos Totales (Pagados): 697,119.67 €
Unidades Vendidas:          23,487
Total Pedidos Válidos:      7,998
Ticket Medio Global:        87.16 €


### Análisis del Reto 8 (KPIs Globales del Export)
* En el conjunto completo de datos limpios (`7.998` pedidos con líneas válidas y `23.487` unidades), el ingreso pagado en caja asciende a **`697.119,67 €`**, con un **ticket medio global de `87,16 €`**.

---
## Reto 9 (Día 2 — Tarde): El Imprevisto de la Tarde — Precio Catálogo vs. Precio Pagado en Caja
Resolvemos la discrepancia que confundía a los departamentos: cuánto valían las prendas a **precio de catálogo** frente a cuánto **entró realmente en caja** tras aplicar los descuentos promocionales, tanto en el total exportado como específicamente en la **Campaña de Rebajas de Verano (`campana_id == 1`)**.

In [10]:
# =============================================================================
# RETO 9 (DÍA 2): CATÁLOGO (BRUTO) VS. PAGADO (CAJA) Y DESCUENTO TOTAL
# =============================================================================
ingreso_catalogo = (lineas['cantidad'] * lineas['precio_unitario']).sum()
ingreso_pagado = lineas['importe_linea'].sum()
descuento_total_euros = ingreso_catalogo - ingreso_pagado
pct_descuento_medio = (descuento_total_euros / ingreso_catalogo) * 100

print('=== 1. TOTAL EXPORTADO (SEMANA PREVIA + CAMPAÑA) ===')
print(f'Ingreso a precio Catálogo:  {ingreso_catalogo:,.2f} €')
print(f'Ingreso Real Pagado (Caja): {ingreso_pagado:,.2f} €')
print(f'Total Descuento Concedido:  {descuento_total_euros:,.2f} € ({pct_descuento_medio:.1f}%)\n')

# Desglose exclusivo de la Campaña Rebajas de Verano (campana_id == 1)
lp_camp = lineas.merge(pedidos[['pedido_id', 'campana_id', 'fecha_pedido']], on='pedido_id', how='inner')
verano_only = lp_camp[lp_camp['campana_id'] == 1]
cat_verano = (verano_only['cantidad'] * verano_only['precio_unitario']).sum()
pag_verano = verano_only['importe_linea'].sum()
desc_verano = cat_verano - pag_verano
pct_desc_verano = (desc_verano / cat_verano) * 100

print('=== 2. SOLO CAMPAÑA REBAJAS DE VERANO 2024 (campana_id == 1) ===')
print(f'Ingreso a precio Catálogo:  {cat_verano:,.2f} €')
print(f'Ingreso Real Pagado (Caja): {pag_verano:,.2f} €')
print(f'Total Descuento Concedido:  {desc_verano:,.2f} € ({pct_desc_verano:.2f}%)')

=== 1. TOTAL EXPORTADO (SEMANA PREVIA + CAMPAÑA) ===
Ingreso a precio Catálogo:  876,633.70 €
Ingreso Real Pagado (Caja): 697,119.67 €
Total Descuento Concedido:  179,514.03 € (20.5%)



=== 2. SOLO CAMPAÑA REBAJAS DE VERANO 2024 (campana_id == 1) ===
Ingreso a precio Catálogo:  748,731.02 €
Ingreso Real Pagado (Caja): 572,301.82 €
Total Descuento Concedido:  176,429.20 € (23.56%)


### Análisis del Reto 9 («¿Cuál es el bueno? Los dos, con su etiqueta al lado»)
* Aquí aparecen exactamente las dos cifras de la conversación con Diego:
  * **En la Campaña de Rebajas de Verano (`campana_id == 1`):** el valor a **precio de catálogo fue de `748.731,02 €`**, pero tras aplicar un **descuento medio efectivo del `23,56 %` (`-176.429,20 €`)**, lo que **realmente entró en caja fueron `572.301,82 €`**.
  * Lección clave de negocio: nunca presentar una cifra de «ventas» sin su apellido (*a precio catálogo*, *pagado en caja* o *neto de devoluciones*).

---
## Reto 10 (Día 2 — Tarde): El Efecto Base (Semana Previa vs. Rebajas)
Un número aislado no dice nada sin una línea base comparable. Comparamos la **Semana Previa sin promoción (`1.179` pedidos orgánicos)** con la primera semana y el promedio semanal de las **Rebajas de Verano**.

In [11]:
# =============================================================================
# RETO 10 (DÍA 2): EFECTO BASE (SEMANA PREVIA VS. CAMPAÑA DE REBAJAS)
# =============================================================================
lp = lineas.merge(pedidos[['pedido_id', 'fecha_pedido', 'campana_id']], on='pedido_id', how='inner')
base = lp[lp['campana_id'].isna()]  # Semana previa sin campaña (1.179 pedidos)
rebajas_sem1 = lp[(lp['fecha_pedido'] >= '2024-07-01') & (lp['fecha_pedido'] <= '2024-07-07')]
rebajas_total = lp[lp['campana_id'] == 1]

def reporte_periodo(df, nombre):
    ing = df['importe_linea'].sum()
    uni = df['cantidad'].sum()
    ped = df['pedido_id'].nunique()
    tm = ing / ped if ped else 0
    prendas_ped = uni / ped if ped else 0
    print(f'--- {nombre} ---')
    print(f'Ingresos Pagados: {ing:,.2f} €')
    print(f'Unidades:         {uni:,}')
    print(f'Pedidos:          {ped:,}')
    print(f'Ticket Medio:     {tm:,.2f} €  |  Prendas/pedido: {prendas_ped:.2f}\n')

reporte_periodo(base, 'SEMANA PREVIA (Línea Base sin promoción)')
reporte_periodo(rebajas_sem1, 'SEMANA 1 DE REBAJAS (01/07 a 07/07)')
reporte_periodo(rebajas_total, 'TOTAL CAMPAÑA REBAJAS VERANO (3 semanas, campana_id == 1)')

--- SEMANA PREVIA (Línea Base sin promoción) ---
Ingresos Pagados: 124,817.85 €
Unidades:         3,479
Pedidos:          1,179
Ticket Medio:     105.87 €  |  Prendas/pedido: 2.95

--- SEMANA 1 DE REBAJAS (01/07 a 07/07) ---
Ingresos Pagados: 138,860.42 €
Unidades:         4,854
Pedidos:          1,640
Ticket Medio:     84.67 €  |  Prendas/pedido: 2.96

--- TOTAL CAMPAÑA REBAJAS VERANO (3 semanas, campana_id == 1) ---
Ingresos Pagados: 572,301.82 €
Unidades:         20,008
Pedidos:          6,819
Ticket Medio:     83.93 €  |  Prendas/pedido: 2.93



### Análisis del Reto 10 (Efecto Base y Comportamiento del Ticket Medio)
* **Por qué sube la facturación mientras baja el Ticket Medio:**
  * En la **Semana Previa (línea base)** entraron `1.179` pedidos y `3.479` unidades por **`124.817,85 €`**, con un **Ticket Medio de `105,87 €`** (compras sin descuento).
  * Durante la **Campaña de Rebajas (`campana_id == 1`)**, el volumen se disparó a **`6.819` pedidos** y **`20.008` unidades** (**`572.301,82 €` pagados en caja**, a un ritmo medio de `190.767 €`/semana), mientras que el **Ticket Medio bajó a `83,93 €`** (`-20,7 %`).
  * Como explica Diego, que el ticket medio baje en rebajas **no es una alarma, sino el efecto mecánico del descuento (`23,56 %` medio)**: entra mucha más gente y compra casi la misma cantidad de prendas por carrito (`2,93` uds/pedido vs `2,95` en la semana base), pero paga menos por unidad.

---
## Cierre del Día 2: Exportación de Datasets Limpios a `data/clean/`
Guardamos las tablas saneadas en `data/clean/` para que los análisis de los Días 3, 4 y 5 trabajen sobre una única fuente de verdad verificada.

In [12]:
# =============================================================================
# CIERRE DÍA 2: EXPORTAR DATASETS LIMPIOS A DATA/CLEAN/
# =============================================================================
os.makedirs('data/clean', exist_ok=True)

lineas.to_csv('data/clean/lineas_limpias.csv', index=False)
pedidos.to_csv('data/clean/pedidos_limpios.csv', index=False)
devoluciones.to_csv('data/clean/devoluciones_limpias.csv', index=False)

print("¡Datasets limpios exportados exitosamente a 'data/clean/'!")
print('Archivos generados:', sorted(os.listdir('data/clean')))

¡Datasets limpios exportados exitosamente a 'data/clean/'!


Archivos generados: ['brightcart_master_verano_2024.csv', 'devoluciones_limpias.csv', 'lineas_limpias.csv', 'pedidos_limpios.csv']


### Conclusión Ejecutiva del Día 2
1. **Datos saneados y auditables (`data/clean/`):** Se depuraron `139` duplicados exactos, `15` líneas con cantidad `<= 0` y `42` devoluciones huérfanas, quedando **`19.986` líneas de venta válidas**, **`8.000` pedidos** y **`5.516` devoluciones legítimas**.
2. **Conciliación de ingresos de la campaña (`campana_id == 1`):** La campaña movió **`748.731,02 €` a precio de catálogo**, concedió **`176.429,20 €` en descuentos (`23,56 %`)** e ingresó **`572.301,82 €` reales en caja** a través de **`6.819` pedidos** y **`20.008` prendas**.
3. **Siguiente paso (Día 3):** Bajar del *«cuánto»* al *«qué»* — descubrir qué categorías, qué prendas específicas y qué canales de marketing impulsaron esos `572.301,82 €`.